# HAM10000 Binary Classification with ResNet50, Grad-CAM++, and SHAP

This notebook trains a pretrained ResNet50 model for binary melanoma classification on HAM10000. It addresses class imbalance with weighted focal binary cross-entropy, tunes the decision threshold on the validation split, and examines model explanations with Grad-CAM++ and SHAP across selected training checkpoints.

### Recorded results

- **Test accuracy:** 91.75%.
- **Test precision:** 60.80%.
- **Test recall:** 72.46%.
- **Test F1-score:** 66.12%.
- **Test AUC:** 92.57%.
- **Confusion matrix:** TN 1,258; FP 78; FN 46; TP 121.

> Notebook outputs and execution counters are intentionally cleared. Configure the Google Drive dataset path before running the notebook in Google Colab.


## Environment Setup


In [ ]:
!pip -q install scikit-learn opencv-python grad-cam shap


## Imports


In [ ]:
import os, copy, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFile

from tqdm.auto import tqdm

import cv2
import shap

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
from torchvision.models import resnet50, ResNet50_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True


## Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Configuration


In [ ]:
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 16
HEAD_EPOCHS = 2
FULL_EPOCHS = 10
TOTAL_EPOCHS = HEAD_EPOCHS + FULL_EPOCHS

LR_HEAD = 1e-3
LR_FULL = 8e-5
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
PATIENCE = 4

USE_FOCAL_BCE = True
FOCAL_GAMMA = 2.0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA_ROOT = "/content/drive/MyDrive/HAM_mirror"
CSV_PATH = os.path.join(DATA_ROOT, "GroundTruth.csv")
IMG_DIR = os.path.join(DATA_ROOT, "images")

SAVE_DIR = "/content/ham_resnet50_binary_v4_xai"
os.makedirs(SAVE_DIR, exist_ok=True)

print("DEVICE:", DEVICE)
print("CSV exists:", os.path.exists(CSV_PATH))
print("IMG_DIR exists:", os.path.exists(IMG_DIR))


## Reproducibility


In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)


## Dataset Loading


In [ ]:
df = pd.read_csv(CSV_PATH)
print("Columns:", df.columns.tolist())
display(df.head())

assert "image" in df.columns, "Missing 'image' column"
assert "MEL" in df.columns, "Missing 'MEL' column"

df["target"] = df["MEL"].astype(int)

def make_img_path(x):
    x = str(x)
    if x.lower().endswith((".jpg", ".png", ".jpeg")):
        return os.path.join(IMG_DIR, x)
    return os.path.join(IMG_DIR, x + ".jpg")

df["img_path"] = df["image"].apply(make_img_path)
df = df[df["img_path"].apply(os.path.exists)].reset_index(drop=True)

print("Usable rows:", len(df))
print(df["target"].value_counts())


## Stratified Data Splits


In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    stratify=df["target"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["target"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", len(train_df), train_df["target"].value_counts().to_dict())
print("Val:", len(val_df), val_df["target"].value_counts().to_dict())
print("Test:", len(test_df), test_df["target"].value_counts().to_dict())


## Image Transforms


In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.08, contrast=0.08, saturation=0.05, hue=0.02),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN.tolist(), IMAGENET_STD.tolist()),
])

eval_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN.tolist(), IMAGENET_STD.tolist()),
])


## Dataset and Data Loaders


In [ ]:
class HAMBinaryDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["img_path"]).convert("RGB")
        x = self.transform(img)
        y = torch.tensor(float(row["target"]), dtype=torch.float32)
        return x, y, row["img_path"]

train_ds = HAMBinaryDataset(train_df, train_tf)
val_ds   = HAMBinaryDataset(val_df, eval_tf)
test_ds  = HAMBinaryDataset(test_df, eval_tf)

PIN_MEMORY = torch.cuda.is_available()

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)


## ResNet50 Model


In [ ]:
def load_checkpoint(path):
    try:
        return torch.load(path, map_location=DEVICE, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=DEVICE)

def build_model():
    weights = ResNet50_Weights.IMAGENET1K_V2
    m = resnet50(weights=weights)
    in_features = m.fc.in_features
    m.fc = nn.Sequential(
        nn.Dropout(0.35),
        nn.Linear(in_features, 1)
    )
    return m.to(DEVICE)

model = build_model()

def freeze_backbone(model):
    for param in model.parameters():
        param.requires_grad = False
    for param in model.fc.parameters():
        param.requires_grad = True

def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True

freeze_backbone(model)


## Loss Function


In [ ]:
num_neg = int((train_df["target"] == 0).sum())
num_pos = int((train_df["target"] == 1).sum())
pos_weight_value = num_neg / max(num_pos, 1)

base_bce = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([pos_weight_value], dtype=torch.float32, device=DEVICE),
    reduction="none"
)

def criterion(logits, targets):
    bce = base_bce(logits, targets)
    if USE_FOCAL_BCE:
        probs = torch.sigmoid(logits)
        pt = torch.where(targets == 1, probs, 1 - probs)
        focal_factor = (1 - pt).pow(FOCAL_GAMMA)
        loss = focal_factor * bce
        return loss.mean()
    return bce.mean()

print("pos_weight:", pos_weight_value)
print("USE_FOCAL_BCE:", USE_FOCAL_BCE)


## Optimizer and Scheduler


In [ ]:
def make_optimizer(model, lr):
    params = [p for p in model.parameters() if p.requires_grad]
    return torch.optim.AdamW(params, lr=lr, weight_decay=WEIGHT_DECAY)

optimizer = make_optimizer(model, LR_HEAD)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=1
)

scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())


## Evaluation Utilities


In [ ]:
@torch.no_grad()
def collect_predictions(loader, threshold=0.5):
    model.eval()
    all_probs, all_preds, all_true = [], [], []
    total_loss = 0.0
    n = 0

    for xb, yb, _ in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)

        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            logits = model(xb).squeeze(1)
            loss = criterion(logits, yb)

        probs = torch.sigmoid(logits)
        preds = (probs >= threshold).float()

        bs = xb.size(0)
        total_loss += loss.item() * bs
        n += bs

        all_probs.append(probs.cpu().numpy())
        all_preds.append(preds.cpu().numpy())
        all_true.append(yb.cpu().numpy())

    y_prob = np.concatenate(all_probs)
    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_true)

    return {
        "loss": total_loss / max(n, 1),
        "y_true": y_true,
        "y_prob": y_prob,
        "y_pred": y_pred,
        "acc": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auc": roc_auc_score(y_true, y_prob)
    }

def find_best_threshold(y_true, y_prob):
    rows = []
    best_thr = 0.5
    best_f1 = -1

    for thr in np.arange(0.10, 0.91, 0.01):
        y_pred = (y_prob >= thr).astype(int)
        accv = accuracy_score(y_true, y_pred)
        precv = precision_score(y_true, y_pred, zero_division=0)
        recv = recall_score(y_true, y_pred, zero_division=0)
        f1v = f1_score(y_true, y_pred, zero_division=0)

        rows.append((thr, accv, precv, recv, f1v))
        if f1v > best_f1:
            best_f1 = f1v
            best_thr = float(thr)

    thr_df = pd.DataFrame(rows, columns=["threshold", "acc", "precision", "recall", "f1"])
    return best_thr, best_f1, thr_df.sort_values("f1", ascending=False)


## Training


In [ ]:
best_val_auc = -1
best_epoch = -1
bad_epochs = 0
history = []

for epoch in range(1, TOTAL_EPOCHS + 1):

    if epoch == HEAD_EPOCHS + 1:
        print("\n--- Unfreezing full model ---")
        unfreeze_all(model)
        optimizer = make_optimizer(model, LR_FULL)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="max", factor=0.5, patience=1
        )

    model.train()
    running_loss = 0.0
    n = 0

    pbar = tqdm(train_loader, leave=False)
    for xb, yb, _ in pbar:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            logits = model(xb).squeeze(1)
            loss = criterion(logits, yb)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        bs = xb.size(0)
        running_loss += loss.item() * bs
        n += bs

    train_loss = running_loss / max(n, 1)
    val_res = collect_predictions(val_loader, threshold=0.5)
    scheduler.step(val_res["auc"])

    print(
        f"Epoch {epoch}/{TOTAL_EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_res['loss']:.4f} | "
        f"val_acc={val_res['acc']:.4f} | "
        f"val_f1={val_res['f1']:.4f} | "
        f"val_auc={val_res['auc']:.4f}"
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_res["loss"],
        "val_acc": val_res["acc"],
        "val_f1": val_res["f1"],
        "val_auc": val_res["auc"]
    })

    # save every epoch
    torch.save(model.state_dict(), os.path.join(SAVE_DIR, f"epoch_{epoch}.pth"))

    if val_res["auc"] > best_val_auc:
        best_val_auc = val_res["auc"]
        best_epoch = epoch
        bad_epochs = 0
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, "best_model.pth"))
        print(f"Saved best model at epoch {epoch} with val_auc={best_val_auc:.4f}")
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

history_df = pd.DataFrame(history)
display(history_df)


## Validation Threshold Selection


In [ ]:
model.load_state_dict(load_checkpoint(os.path.join(SAVE_DIR, "best_model.pth")))

val_res = collect_predictions(val_loader, threshold=0.5)
best_thr, best_val_f1, thr_df = find_best_threshold(val_res["y_true"], val_res["y_prob"])

print("Best threshold:", best_thr)
print("Best val F1:", best_val_f1)
display(thr_df.head(10))


## Test Evaluation


In [ ]:
test_res = collect_predictions(test_loader, threshold=best_thr)

print("\n=== TEST RESULTS ===")
print("Best epoch:", best_epoch)
print("Chosen threshold:", best_thr)
print("Test Loss:", test_res["loss"])
print("Test Accuracy:", test_res["acc"])
print("Test Precision:", test_res["precision"])
print("Test Recall:", test_res["recall"])
print("Test F1:", test_res["f1"])
print("Test AUC:", test_res["auc"])

print("\nConfusion Matrix:")
print(confusion_matrix(test_res["y_true"], test_res["y_pred"]))

print("\nClassification Report:")
print(classification_report(
    test_res["y_true"],
    test_res["y_pred"],
    target_names=["non-melanoma", "melanoma"],
    digits=4
))


## Training and Validation Curves


In [ ]:
plt.figure(figsize=(7,4))
plt.plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
plt.plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training / Validation Loss")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7,4))
plt.plot(history_df["epoch"], history_df["val_auc"], label="val_auc")
plt.plot(history_df["epoch"], history_df["val_f1"], label="val_f1")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Validation AUC / F1")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7,4))
thr_sorted = thr_df.sort_values("threshold")
plt.plot(thr_sorted["threshold"], thr_sorted["precision"], label="precision")
plt.plot(thr_sorted["threshold"], thr_sorted["recall"], label="recall")
plt.plot(thr_sorted["threshold"], thr_sorted["f1"], label="f1")
plt.axvline(best_thr, linestyle="--")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Validation Threshold Sweep")
plt.legend()
plt.tight_layout()
plt.show()


## Test Prediction Export


In [ ]:
test_out = test_df.copy().reset_index(drop=True)
test_out["y_true"] = test_res["y_true"]
test_out["y_prob"] = test_res["y_prob"]
test_out["y_pred"] = test_res["y_pred"]
test_out["correct"] = (test_out["y_true"] == test_out["y_pred"]).astype(int)

csv_out = os.path.join(SAVE_DIR, "test_predictions.csv")
test_out.to_csv(csv_out, index=False)
print("Saved:", csv_out)


## Explainability Utilities


In [ ]:
def load_raw_image(path, img_size=IMG_SIZE):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_AREA)
    return img

def preprocess_numpy_image(img_rgb):
    x = img_rgb.astype(np.float32) / 255.0
    x = (x - IMAGENET_MEAN) / IMAGENET_STD
    x = np.transpose(x, (2, 0, 1))
    x = torch.tensor(x, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    return x

def predict_prob_from_model(model_obj, img_path, threshold):
    img_rgb = load_raw_image(img_path, IMG_SIZE)
    x = preprocess_numpy_image(img_rgb)
    model_obj.eval()
    with torch.no_grad():
        logit = model_obj(x).squeeze(1)
        prob = torch.sigmoid(logit).item()
    pred = int(prob >= threshold)
    return img_rgb, prob, pred

def load_model_ckpt(ckpt_path):
    m = build_model()
    m.load_state_dict(load_checkpoint(ckpt_path))
    m.eval()
    return m


## Representative Case Selection


In [ ]:
correct_mel = test_out[(test_out["y_true"] == 1) & (test_out["y_pred"] == 1)].copy()
correct_non = test_out[(test_out["y_true"] == 0) & (test_out["y_pred"] == 0)].copy()
wrong_cases = test_out[test_out["correct"] == 0].copy()

idx_correct_mel = None
idx_correct_non = None
idx_wrong = None

if len(correct_mel) > 0:
    idx_correct_mel = correct_mel.sort_values("y_prob", ascending=False).index[0]

if len(correct_non) > 0:
    idx_correct_non = correct_non.sort_values("y_prob", ascending=True).index[0]

if len(wrong_cases) > 0:
    wrong_cases = wrong_cases.copy()
    wrong_cases["confidence"] = np.where(
        wrong_cases["y_pred"] == 1,
        wrong_cases["y_prob"],
        1 - wrong_cases["y_prob"]
    )
    idx_wrong = wrong_cases.sort_values("confidence", ascending=False).index[0]

print("idx_correct_mel:", idx_correct_mel)
print("idx_correct_non:", idx_correct_non)
print("idx_wrong:", idx_wrong)

# choose one case here
# options: idx_correct_mel, idx_correct_non, idx_wrong
CASE_IDX = idx_correct_mel if idx_correct_mel is not None else idx_wrong
if CASE_IDX is None:
    raise RuntimeError("No suitable test sample was found for XAI visualization.")

selected_row = test_out.loc[CASE_IDX]
SELECTED_IMG_PATH = selected_row["img_path"]
SELECTED_TRUE = int(selected_row["y_true"])

print("Selected image:", SELECTED_IMG_PATH)
print("Selected true label:", SELECTED_TRUE)


## Grad-CAM++ Across Training Checkpoints


In [ ]:
def show_gradcam_for_ckpt(ckpt_path, title, threshold=best_thr):
    model_obj = load_model_ckpt(ckpt_path)
    img_rgb, prob, pred = predict_prob_from_model(model_obj, SELECTED_IMG_PATH, threshold)

    input_tensor = preprocess_numpy_image(img_rgb)
    rgb_float = img_rgb.astype(np.float32) / 255.0

    target_layers = [model_obj.layer4[-1].conv3]
    cam = GradCAMPlusPlus(model=model_obj, target_layers=target_layers)

    grayscale_cam = cam(input_tensor=input_tensor)[0]
    overlay = show_cam_on_image(rgb_float, grayscale_cam, use_rgb=True, image_weight=0.75)

    plt.figure(figsize=(12,4))

    plt.subplot(1,3,1)
    plt.imshow(img_rgb)
    plt.title(f"{title}\nImage\ntrue={SELECTED_TRUE} pred={pred}\nprob={prob:.4f}")
    plt.axis("off")

    plt.subplot(1,3,2)
    plt.imshow(grayscale_cam, cmap="jet")
    plt.title("Grad-CAM++")
    plt.axis("off")

    plt.subplot(1,3,3)
    plt.imshow(overlay)
    plt.title("Overlay")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

EPOCHS_TO_SHOW = [1, 4, 8]

for ep in EPOCHS_TO_SHOW:
    ckpt_path = os.path.join(SAVE_DIR, f"epoch_{ep}.pth")
    if os.path.exists(ckpt_path):
        print(f"\n================ EPOCH {ep} | GRAD-CAM++ ================\n")
        show_gradcam_for_ckpt(ckpt_path, f"Epoch {ep}")
    else:
        print(f"Checkpoint not found: {ckpt_path}")

best_ckpt_path = os.path.join(SAVE_DIR, "best_model.pth")
if os.path.exists(best_ckpt_path):
    print(f"\n================ BEST MODEL | GRAD-CAM++ ================\n")
    show_gradcam_for_ckpt(best_ckpt_path, "Best Model")


## SHAP Across Training Checkpoints


In [ ]:
def make_shap_predict(model_obj):
    def _predict(batch_numpy):
        xs = []
        for img in batch_numpy:
            img = img.astype(np.float32) / 255.0
            img = (img - IMAGENET_MEAN) / IMAGENET_STD
            img = np.transpose(img, (2, 0, 1))
            xs.append(img)
        x = torch.tensor(np.stack(xs), dtype=torch.float32).to(DEVICE)

        model_obj.eval()
        with torch.no_grad():
            logits = model_obj(x).squeeze(1)
            probs = torch.sigmoid(logits).cpu().numpy()
        return probs.reshape(-1, 1)
    return _predict

def show_shap_for_ckpt(ckpt_path, title, threshold=best_thr):
    model_obj = load_model_ckpt(ckpt_path)
    img_rgb, prob, pred = predict_prob_from_model(model_obj, SELECTED_IMG_PATH, threshold)

    masker = shap.maskers.Image("blur(32,32)", (IMG_SIZE, IMG_SIZE, 3))
    explainer = shap.Explainer(make_shap_predict(model_obj), masker)

    shap_values = explainer(
        np.expand_dims(img_rgb, axis=0),
        max_evals=500,
        batch_size=16
    )

    plt.figure(figsize=(5,5))
    plt.imshow(img_rgb)
    plt.title(f"{title}\nImage\ntrue={SELECTED_TRUE} pred={pred}\nprob={prob:.4f}")
    plt.axis("off")
    plt.show()

    # IMPORTANT: separate plot, not subplot
    shap.image_plot(shap_values)

for ep in EPOCHS_TO_SHOW:
    ckpt_path = os.path.join(SAVE_DIR, f"epoch_{ep}.pth")
    if os.path.exists(ckpt_path):
        print(f"\n================ EPOCH {ep} | SHAP ================\n")
        show_shap_for_ckpt(ckpt_path, f"Epoch {ep}")
    else:
        print(f"Checkpoint not found: {ckpt_path}")

if os.path.exists(best_ckpt_path):
    print(f"\n================ BEST MODEL | SHAP ================\n")
    show_shap_for_ckpt(best_ckpt_path, "Best Model")

print("Done.")
print("Outputs saved in:", SAVE_DIR)


## Pixel-Level SHAP Explanation


In [ ]:
IMG_SIZE = 224
BACKGROUND_N = 8
NSAMPLES = 50

model = model.to(DEVICE)
model.eval()

def find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"None of the expected columns were found: {candidates}")

path_col = find_col(
    test_df,
    ["image_path", "img_path", "path", "filepath", "file_path", "full_path"]
)

label_col = find_col(
    test_df,
    ["label", "target", "binary_label", "melanoma", "class"]
)

print("Path column:", path_col)
print("Label column:", label_col)

mel_df = test_df[test_df[label_col].astype(int) == 1].reset_index(drop=True)

if len(mel_df) == 0:
    raise ValueError("No melanoma samples with label 1 were found in test_df.")

mel_row = mel_df.sample(1, random_state=42).iloc[0]
mel_path = mel_row[path_col]

print("Melanoma image selected:")
print(mel_path)

mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

preprocess = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=mean, std=std)
])

def load_image_tensor(path):
    img = Image.open(path).convert("RGB")
    x = preprocess(img)
    return x

def load_image_for_plot(path):
    img = Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    return np.array(img) / 255.0

bg_df = train_df.sample(
    min(BACKGROUND_N, len(train_df)),
    random_state=42
).reset_index(drop=True)

background_tensors = []

for p in bg_df[path_col].tolist():
    try:
        background_tensors.append(load_image_tensor(p))
    except Exception as e:
        print("Problem loading background image:", p)
        print(e)

background = torch.stack(background_tensors).to(DEVICE)

# class 0 = non-melanoma
# class 1 = melanoma

class BinaryProbWrapper(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def forward(self, x):
        logits = self.base_model(x)

        if isinstance(logits, tuple):
            logits = logits[0]

        if logits.ndim == 1:
            logits = logits.unsqueeze(1)

        if logits.shape[1] == 1:
            p_mel = torch.sigmoid(logits)
            p_nonmel = 1 - p_mel
            return torch.cat([p_nonmel, p_mel], dim=1)

        return torch.softmax(logits, dim=1)

wrapped_model = BinaryProbWrapper(model).to(DEVICE)
wrapped_model.eval()

x = load_image_tensor(mel_path).unsqueeze(0).to(DEVICE)
original_img = load_image_for_plot(mel_path)

with torch.no_grad():
    probs = wrapped_model(x)
    melanoma_prob = probs[0, 1].item()

print(f"Melanoma probability: {melanoma_prob:.4f}")

explainer = shap.GradientExplainer(wrapped_model, background)

shap_values = explainer.shap_values(
    x,
    nsamples=NSAMPLES
)

if isinstance(shap_values, list):
    sv = shap_values[1][0]   # class 1 = melanoma
else:
    if shap_values.ndim == 5:
        sv = shap_values[0, ..., 1]
    else:
        sv = shap_values[0]

if sv.shape[0] == 3:
    shap_map = sv.sum(axis=0)
elif sv.shape[-1] == 3:
    shap_map = sv.sum(axis=-1)
else:
    shap_map = np.squeeze(sv)

abs_max = np.percentile(np.abs(shap_map), 99)

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(original_img)
plt.title("Original melanoma image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(shap_map, cmap="bwr", vmin=-abs_max, vmax=abs_max, interpolation="nearest")
plt.title("Pixel-level SHAP map")
plt.colorbar(fraction=0.046, pad=0.04)
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(original_img)
plt.imshow(shap_map, cmap="bwr", vmin=-abs_max, vmax=abs_max, alpha=0.45, interpolation="nearest")
plt.title(f"SHAP overlay | melanoma prob = {melanoma_prob:.3f}")
plt.axis("off")

plt.tight_layout()

save_path = "/content/drive/MyDrive/shap_melanoma_pixel_level.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved SHAP image at:")
print(save_path)


## SHAP Evolution Across Training Checkpoints


In [ ]:
IMG_SIZE = 224
BACKGROUND_N = 4
NSAMPLES = 25

CHECKPOINTS = {
    "Epoch 1": os.path.join(SAVE_DIR, "epoch_1.pth"),
    "Epoch 4": os.path.join(SAVE_DIR, "epoch_4.pth"),
    "Epoch 8": os.path.join(SAVE_DIR, "epoch_8.pth"),
    "Best": os.path.join(SAVE_DIR, "best_model.pth"),
}

def find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"None of the expected columns were found: {candidates}")

path_col = find_col(
    test_df,
    ["image_path", "img_path", "path", "filepath", "file_path", "full_path"]
)

label_col = find_col(
    test_df,
    ["label", "target", "binary_label", "melanoma", "class"]
)

print("Path column:", path_col)
print("Label column:", label_col)

mel_df = test_df[test_df[label_col].astype(int) == 1].reset_index(drop=True)

if len(mel_df) == 0:
    raise ValueError("No melanoma samples with label 1 were found in test_df.")

mel_row = mel_df.sample(1, random_state=42).iloc[0]
mel_path = mel_row[path_col]

print("Selected melanoma image:")
print(mel_path)

# Transforms

mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

preprocess = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=mean, std=std)
])

def load_image_tensor(path):
    img = Image.open(path).convert("RGB")
    return preprocess(img)

def load_image_for_plot(path):
    img = Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    return np.array(img) / 255.0

x = load_image_tensor(mel_path).unsqueeze(0).to(DEVICE)
original_img = load_image_for_plot(mel_path)

bg_df = train_df.sample(
    min(BACKGROUND_N, len(train_df)),
    random_state=42
).reset_index(drop=True)

background_tensors = []

for p in bg_df[path_col].tolist():
    try:
        background_tensors.append(load_image_tensor(p))
    except Exception as e:
        print("Problem loading background image:", p)
        print(e)

background = torch.stack(background_tensors).to(DEVICE)

print("Background shape:", background.shape)

# 0 = non-melanoma, 1 = melanoma

class BinaryProbWrapper(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def forward(self, x):
        logits = self.base_model(x)

        if isinstance(logits, tuple):
            logits = logits[0]

        if logits.ndim == 1:
            logits = logits.unsqueeze(1)

        if logits.shape[1] == 1:
            p_mel = torch.sigmoid(logits)
            p_nonmel = 1 - p_mel
            return torch.cat([p_nonmel, p_mel], dim=1)

        return torch.softmax(logits, dim=1)

def clean_state_dict(state):
    if isinstance(state, dict):
        if "model_state_dict" in state:
            state = state["model_state_dict"]
        elif "state_dict" in state:
            state = state["state_dict"]

    cleaned = {}
    for k, v in state.items():
        new_k = k.replace("module.", "")
        cleaned[new_k] = v

    return cleaned

def compute_shap_for_checkpoint(checkpoint_path, checkpoint_name):
    print(f"\nProcessing: {checkpoint_name}")
    print(checkpoint_path)

    if not os.path.exists(checkpoint_path):
        print(f"Checkpoint not found: {checkpoint_path}")
        return None

    epoch_model = copy.deepcopy(model)
    state = load_checkpoint(checkpoint_path)
    state = clean_state_dict(state)

    epoch_model.load_state_dict(state, strict=False)
    epoch_model = epoch_model.to(DEVICE)
    epoch_model.eval()

    wrapped_model = BinaryProbWrapper(epoch_model).to(DEVICE)
    wrapped_model.eval()

    with torch.no_grad():
        probs = wrapped_model(x)
        melanoma_prob = probs[0, 1].item()

    print(f"{checkpoint_name} melanoma probability: {melanoma_prob:.4f}")

    explainer = shap.GradientExplainer(wrapped_model, background)

    shap_values = explainer.shap_values(
        x,
        nsamples=NSAMPLES
    )

    if isinstance(shap_values, list):
        sv = shap_values[1][0]
    else:
        if shap_values.ndim == 5:
            sv = shap_values[0, ..., 1]
        else:
            sv = shap_values[0]

    if sv.shape[0] == 3:
        shap_map = sv.sum(axis=0)
    elif sv.shape[-1] == 3:
        shap_map = sv.sum(axis=-1)
    else:
        shap_map = np.squeeze(sv)

    return {
        "name": checkpoint_name,
        "prob": melanoma_prob,
        "shap_map": shap_map
    }

results = []

for name, ckpt_path in CHECKPOINTS.items():
    res = compute_shap_for_checkpoint(ckpt_path, name)
    if res is not None:
        results.append(res)

if len(results) == 0:
    raise ValueError("No checkpoint was found. Verify the paths in CHECKPOINTS.")

all_values = np.concatenate([r["shap_map"].flatten() for r in results])
abs_max = np.percentile(np.abs(all_values), 99)

# Plot: Original + SHAP overlays across epochs

n = len(results)

plt.figure(figsize=(4 * (n + 1), 5))

plt.subplot(1, n + 1, 1)
plt.imshow(original_img)
plt.title("Original\nMelanoma image")
plt.axis("off")

for i, r in enumerate(results):
    plt.subplot(1, n + 1, i + 2)
    plt.imshow(original_img)
    plt.imshow(
        r["shap_map"],
        cmap="bwr",
        vmin=-abs_max,
        vmax=abs_max,
        alpha=0.45,
        interpolation="nearest"
    )
    plt.title(f"{r['name']}\nP(mel)={r['prob']:.3f}")
    plt.axis("off")

plt.tight_layout()

save_path = "/content/drive/MyDrive/shap_across_epochs_melanoma.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved figure:")
print(save_path)

plt.figure(figsize=(4 * n, 5))

for i, r in enumerate(results):
    plt.subplot(1, n, i + 1)
    plt.imshow(
        r["shap_map"],
        cmap="bwr",
        vmin=-abs_max,
        vmax=abs_max,
        interpolation="nearest"
    )
    plt.title(f"{r['name']}\nP(mel)={r['prob']:.3f}")
    plt.axis("off")
    plt.colorbar(fraction=0.046, pad=0.04)

plt.tight_layout()

save_path_maps = "/content/drive/MyDrive/shap_maps_across_epochs_melanoma.png"
plt.savefig(save_path_maps, dpi=300, bbox_inches="tight")
plt.show()

print("Saved SHAP maps:")
print(save_path_maps)
